In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
)

from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

transactions_data = [
    (1, 1, "2023-07-01", 100.0),
    (2, 1, "2023-07-02", 150.0),
    (2, 2, "2023-07-01", 200.0),
    (3, 3, "2023-07-03", 250.0),
    (-4, 4, "2023-07-04", 300.0),
    (5, 2, "2023-25-01", 350.0),
    (6, 1, "2023-07-02", 400.0),
    (7, 6, "2023-07-01", 450.0),
    (8, 7, "2023-07-03", 500.0),
    (9, -8, "2023-07-04", 550.0),
]

transactions_schema = StructType(
    [
        StructField("TransactionID", IntegerType(), True),
        StructField("ClientID", IntegerType(), True),
        StructField("Date", StringType(), True),
        StructField("Amount", DoubleType(), True),
    ]
)

df_transactions = spark.createDataFrame(transactions_data, schema=transactions_schema)

clients_data = [
    (1, "Client1", "Tech"),
    (2, "Client2", "Finance"),
    (3, "Client3", "Real Estate"),
    (4, "Client4", "Healthcare"),
    (5, "Client5", "Tech"),
    (1, "Client6", "Finance"),
    (6, "Client7", "Real Estate"),
    (-7, "Client8", "Healthcare"),
    (8, "Client9", "Tech"),
    (2, "Client10", "Finance"),
]

clients_schema = StructType(
    [
        StructField("ClientID", IntegerType(), True),
        StructField("ClientName", StringType(), True),
        StructField("Industry", StringType(), True),
    ]
)

df_clients = spark.createDataFrame(clients_data, schema=clients_schema)

In [9]:
df_transactions = df_transactions.dropDuplicates(subset=["TransactionID"])
df_transactions = df_transactions.filter(col("TransactionID") > 0)
df_transactions.show()

+-------------+--------+----------+------+
|TransactionID|ClientID|      Date|Amount|
+-------------+--------+----------+------+
|            1|       1|2023-07-01| 100.0|
|            2|       1|2023-07-02| 150.0|
|            3|       3|2023-07-03| 250.0|
|            5|       2|2023-25-01| 350.0|
|            6|       1|2023-07-02| 400.0|
|            7|       6|2023-07-01| 450.0|
|            8|       7|2023-07-03| 500.0|
|            9|      -8|2023-07-04| 550.0|
+-------------+--------+----------+------+



In [10]:
df_clients = df_clients.dropDuplicates(subset=["ClientID"])
df_clients = df_clients.filter(col("ClientID") > 0)
df_clients.show()

+--------+----------+-----------+
|ClientID|ClientName|   Industry|
+--------+----------+-----------+
|       1|   Client1|       Tech|
|       2|   Client2|    Finance|
|       3|   Client3|Real Estate|
|       4|   Client4| Healthcare|
|       5|   Client5|       Tech|
|       6|   Client7|Real Estate|
|       8|   Client9|       Tech|
+--------+----------+-----------+



In [13]:
df_joined = df_transactions.join(df_clients, "ClientID", "inner")
df_joined.orderBy("TransactionID").show()

+--------+-------------+----------+------+----------+-----------+
|ClientID|TransactionID|      Date|Amount|ClientName|   Industry|
+--------+-------------+----------+------+----------+-----------+
|       1|            1|2023-07-01| 100.0|   Client1|       Tech|
|       1|            2|2023-07-02| 150.0|   Client1|       Tech|
|       3|            3|2023-07-03| 250.0|   Client3|Real Estate|
|       2|            5|2023-25-01| 350.0|   Client2|    Finance|
|       1|            6|2023-07-02| 400.0|   Client1|       Tech|
|       6|            7|2023-07-01| 450.0|   Client7|Real Estate|
+--------+-------------+----------+------+----------+-----------+

